In [1]:
!pip install --upgrade "accelerate>=0.26.0"

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from datasets import Dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    classification_report,
    matthews_corrcoef,
    balanced_accuracy_score,
)
from degender_pronoun import degenderizer

/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/usr/local/lib/python3.10/dist-packages/torchvision/transforms/v2/__init__.py:64: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https:/

In [3]:
model_checkpoint = "keshavkumaresan/distilbert-base-uncased-debiased"
data_column = "s1_s2"
preprocessing_type = "all"
batch_size = 16
metric_name = "f1"
labels = ["female", "male"]
num_labels = 2
dataset_path = "data/sentence_sets_trimmed.csv"


In [4]:
degender_pronouns = {
    " mr ": " mx ",
    " mrs ": " mx ",
    " ms ": " mx ",
    " miss ": " mx ",
    " mister ": " mx ",
}

degender_nouns = {
    " man ": " person ",
    " men ": " persons ",
    " woman ": " person ",
    " women ": " persons ",
    " man's ": " person's ",
    " men's ": " person's ",
    " woman's ": " person's ",
    " women's ": " person's ",
    " gentleman ": " person ",
    " lady ": " person ",
    " gentleman's ": " person's ",
    " lady's ": " person's ",
}

In [5]:
def preprocess(df, data_col, p_type="none"):
    if p_type == "none":
        return df
    D = degenderizer()
    df[data_col] = df[data_col].apply(lambda x: D.degender(x) if len(x) > 5 else x)
    for old, new in degender_pronouns.items():
        df[data_col] = df[data_col].str.lower().replace(old, new)
    if p_type == "all":
        for old, new in degender_nouns.items():
            df[data_col] = df[data_col].str.lower().replace(old, new)
    return df

In [28]:
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs["logits"]
#         loss_fct = nn.CrossEntropyLoss(
#             weight=torch.tensor([8.0, 1.0], device=model.device)
#         )
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [7]:
metric = evaluate.load(metric_name)
confusion_metric = evaluate.load("confusion_matrix")

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    cm = confusion_metric.compute(predictions=preds, references=true_labels)
    print("Confusion Matrix:", cm)
    print(classification_report(true_labels, preds, target_names=labels))
    print("MCC:", matthews_corrcoef(true_labels, preds))
    print("Balanced Accuracy:", balanced_accuracy_score(true_labels, preds))
    return metric.compute(predictions=preds, references=true_labels, average="macro")

In [8]:
def preprocess_function(sample):
    return tokenizer(sample[data_column], truncation=True, padding=True)

In [9]:
df = pd.read_csv(dataset_path, encoding="unicode_escape")
df = preprocess(df, data_column, preprocessing_type)

In [10]:
dataset = (
    Dataset.from_pandas(df)
    .rename_column("applicant_gender", "label")
    .class_encode_column("label")
)
dataset = dataset.train_test_split(test_size=0.2)

Casting to class labels:   0%|          | 0/3285 [00:00<?, ? examples/s]

In [11]:
# model_checkpoint = "keshavkumaresan/distilbert-base-uncased-debiased"
model_checkpoint = "distilbert-base-uncased"
# model_checkpoint = "tomhosking/bert-base-uncased-debiased-nli"
# model_checkpoint = "roberta-base"

In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, num_labels=num_labels
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
encoded_dataset = dataset.map(preprocess_function, batched=True)
task = f"nlp-letters-{data_column}-{preprocessing_type}-class-weighted"
model_name = model_checkpoint.split("/")[-1]

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-{task}",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
)

Map:   0%|          | 0/2628 [00:00<?, ? examples/s]

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [30]:
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


/tmp/ipykernel_2807495/3603755969.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  trainer = CustomTrainer(


In [31]:
trainer.train()
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

Epoch,Training Loss,Validation Loss,F1
1,No log,0.509268,0.626521
2,No log,0.472755,0.634451


Confusion Matrix: {'confusion_matrix': array([[ 47, 146],
       [  1, 463]])}
              precision    recall  f1-score   support

      female       0.98      0.24      0.39       193
        male       0.76      1.00      0.86       464

    accuracy                           0.78       657
   macro avg       0.87      0.62      0.63       657
weighted avg       0.82      0.78      0.72       657

MCC: 0.42246266085169676
Balanced Accuracy: 0.6206840718241915
Confusion Matrix: {'confusion_matrix': array([[ 51, 142],
       [  6, 458]])}
              precision    recall  f1-score   support

      female       0.89      0.26      0.41       193
        male       0.76      0.99      0.86       464

    accuracy                           0.77       657
   macro avg       0.83      0.63      0.63       657
weighted avg       0.80      0.77      0.73       657

MCC: 0.40667513239251624
Balanced Accuracy: 0.6256588350902269


Confusion Matrix: {'confusion_matrix': array([[ 51, 142],
       [  6, 458]])}
              precision    recall  f1-score   support

      female       0.89      0.26      0.41       193
        male       0.76      0.99      0.86       464

    accuracy                           0.77       657
   macro avg       0.83      0.63      0.63       657
weighted avg       0.80      0.77      0.73       657

MCC: 0.40667513239251624
Balanced Accuracy: 0.6256588350902269
Evaluation Results: {'eval_loss': 0.4727550446987152, 'eval_f1': 0.6344511278195488, 'eval_runtime': 3.7774, 'eval_samples_per_second': 173.93, 'eval_steps_per_second': 11.119, 'epoch': 2.0}
